# 🚀 MDAAR v2.0: Multi-role Dynamic Architecture for Automated Repositories
### Polyglot Reverse Engineering, Dual-Model Planning/Execution & Persistent Hybrid RAG
---
**What's New in v2.0.0:**
1. **100% Full-File Codebase Coverage:** No 40-file cap or 25k context truncation — lossless `Map-Reduce` analysis over every non-ignored file with `SecretSanitizer` redaction.
2. **Dual-Model Hybrid Architecture (`Planner` + `Executor`):**
   - **Planner Model Settings:** Synthesizes the Master Architecture Blueprint and performs final Quality Audit.
   - **Executor Model Settings:** Always performs heavy 100% file-by-file inspection (`Map`), generates full doc packages (`Reduce`), and answers RAG queries (or select *Same as Planner Model*).
3. **Polyglot Deterministic Extractors:** PHP (Yii2, Laravel, Symfony), Python AST, JS/TS, Java/Kotlin, C#/.NET, Go, Ruby, Rust, Dart/Flutter + SQL DDL/ORM Migrations + Postman + OpenAPI specs.
4. **Repository-Bundled Guardrail Skills:** `core-grounding-qa`, `reverse-engineering`, `architecture-mermaid`, `api-openapi-spec`, and `[TBD]`/`[⚠️ BLIND SPOT]` honesty enforcement.
5. **Automated Quality Gate & L0/L1 Document Register:** Validates Mermaid diagrams, grounds identifiers against extracted symbols, and publishes full packages.
6. **Persistent Hybrid BM25 RAG in `.rag_index/`:** Indexes docs alongside real code FileFactCards, DB schemas, and API endpoints with audience filtering (`Developer`, `Management`, `End-User`).

## 1️⃣ Environment Setup & MDAAR v2.0 Installation
Mount Google Drive for persistent storage and install **MDAAR** along with bundled skills and T4 GPU quantization libraries.

In [ ]:
#@title 1. Initialize Runtime, Mount Drive & Install MDAAR v2.0 { display-mode: "form" }
import os, sys, json, shutil, subprocess
from pathlib import Path

DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ Google Drive mounted successfully.')
    except Exception as e:
        print('⚠️ Drive mount notice (using local storage fallback):', e)

print('⏳ Installing MDAAR v2.0, bundled skills, and T4 quantization tools (transformers, accelerate, bitsandbytes)...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers', 'accelerate', 'bitsandbytes'], check=True)

cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ Installing cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, SkillLoader
skills_count = len(SkillLoader().list_bundled_skills())
print(f'🎉 MDAAR v{__version__} ready with {skills_count} bundled guardrail skills!')

## 2️⃣ Source Repository Configuration (Git)
Configure project name, source code Git URL, and SSH/HTTPS authentication.

In [ ]:
#@title 2. Configure Source Repository { display-mode: "form" }
PROJECT_NAME = "MySystem" #@param {type:"string"}
GIT_SOURCE_URL = "https://github.com/octocat/Hello-World.git" #@param {type:"string"}
SOURCE_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if os.path.exists('/content/drive/MyDrive'):
    BASE_WORKSPACE = Path('/content/drive/MyDrive/AI_Documentation_Workspace')
else:
    BASE_WORKSPACE = Path('/content/AI_Documentation_Workspace')

from mdaar_pipeline import MDAARPipeline, ModelGateway, DualModelManager

pipeline = MDAARPipeline(
    project_name=PROJECT_NAME,
    base_workspace=BASE_WORKSPACE,
    git_source_url=GIT_SOURCE_URL,
    source_auth=SOURCE_AUTH_METHOD
)

if SOURCE_AUTH_METHOD == "SSH":
    source_pub_key = pipeline.setup_source_ssh()
    print('='*70)
    print('🔑 Source Repository SSH Public Key (Deploy Key):')
    print(source_pub_key)
    print('='*70)

print('⏳ Testing connection to Source Repository...')
source_conn = pipeline.test_source_connection()
print(source_conn['message'])
if not source_conn['success'] and SOURCE_AUTH_METHOD == "SSH":
    print('💡 Add the Deploy Key above in Settings -> Deploy Keys, then re-run this cell to verify.')

print(f'✅ Project workspace ready at: {pipeline.project_dir}')
print(f'🔍 Hybrid RAG Index path: {pipeline.rag_index_dir}')

## 3️⃣ External Shared Documentation Repository (Optional)
Push generated docs into a central documentation hub (`projects/<ProjectName>/`).

In [ ]:
#@title 3. Configure External Documentation Repository { display-mode: "form" }
ENABLE_EXTERNAL_DOCS_REPO = False #@param {type:"boolean"}
GIT_DOCS_URL = "" #@param {type:"string"}
DOCS_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if ENABLE_EXTERNAL_DOCS_REPO and GIT_DOCS_URL.strip():
    pipeline.git_docs_url = GIT_DOCS_URL.strip()
    pipeline.docs_auth = DOCS_AUTH_METHOD
    if DOCS_AUTH_METHOD == "SSH":
        docs_pub_key = pipeline.setup_docs_ssh()
        print('='*70)
        print('🔑 Docs Repository SSH Public Key (Deploy Key with Write access):')
        print(docs_pub_key)
        print('='*70)
    print('⏳ Testing connection to External Documentation Repository...')
    docs_conn = pipeline.test_docs_connection()
    print(docs_conn['message'])
    if not docs_conn['success'] and DOCS_AUTH_METHOD == "SSH":
        print('💡 Add the Deploy Key above with Write permission, then re-run this cell to confirm.')
else:
    pipeline.git_docs_url = None
    print('ℹ️ Docs and .rag_index/ will be saved locally in Google Drive/Workspace.')

## 4️⃣ Dual-Model AI Architecture (`Planner Model Settings` + `Executor Model Settings`)
Split planning and execution cleanly or use a single model for both:
- **Planner Model Settings:** Synthesizes the Master Architecture Blueprint & performs Quality Review (ideal for strong cloud models like Gemini or large reasoning models).
- **Executor Model Settings:** Always performs 100% file-by-file codebase inspection (`Map`), authors full packages (`Reduce`), and answers Q&A (ideal for offline quantized 4-bit T4 models — or select `Same as Planner Model`).
> 💡 When pairing two distinct local HuggingFace models on a single Colab T4 (15GB VRAM), `DualModelManager` automatically unloads the inactive model before loading the active one.

In [ ]:
#@title 4a. Planner Model Settings (Master Blueprint & QA Review) { display-mode: "form" }
PLANNER_PROVIDER = "Gemini" #@param ["Gemini", "Local-Offline"]
PLANNER_GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-1.5-pro", "gemini-2.0-flash"]
PLANNER_GEMINI_KEYS_INPUT = "" #@param {type:"string"}
PLANNER_LOCAL_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
PLANNER_QUANTIZATION = "4bit" #@param ["4bit", "None"]
PLANNER_HF_TOKEN_INPUT = "" #@param {type:"string"}

def _resolve_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

planner_keys = [k.strip() for k in PLANNER_GEMINI_KEYS_INPUT.split(',') if k.strip()]
if not planner_keys:
    sec_gem = _resolve_colab_secret('GEMINI_API_KEY')
    if sec_gem:
        planner_keys.append(sec_gem)

planner_hf = PLANNER_HF_TOKEN_INPUT.strip() or _resolve_colab_secret('HF_TOKEN')

planner_gateway = ModelGateway(
    provider=PLANNER_PROVIDER,
    model_name=PLANNER_GEMINI_MODEL,
    api_keys=planner_keys,
    local_model_path=PLANNER_LOCAL_MODEL_NAME,
    hf_token=planner_hf,
    quantization=PLANNER_QUANTIZATION
)
pipeline.planner_gateway = planner_gateway
pipeline.model_manager = DualModelManager(pipeline.planner_gateway, pipeline.executor_gateway)

print(f'✅ Planner Model configured: {planner_gateway.descriptor}')

In [ ]:
#@title 4b. Executor Model Settings (Full-Codebase Inspection & Doc Authoring) { display-mode: "form" }
EXECUTOR_MODE = "Same as Planner Model" #@param ["Same as Planner Model", "Custom Executor Model"]
EXECUTOR_PROVIDER = "Local-Offline" #@param ["Local-Offline", "Gemini"]
EXECUTOR_GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-pro"]
EXECUTOR_GEMINI_KEYS_INPUT = "" #@param {type:"string"}
EXECUTOR_LOCAL_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
EXECUTOR_QUANTIZATION = "4bit" #@param ["4bit", "None"]
EXECUTOR_HF_TOKEN_INPUT = "" #@param {type:"string"}

if EXECUTOR_MODE == "Same as Planner Model":
    executor_gateway = pipeline.planner_gateway
    print(f'ℹ️ Single-model mode active: Executor = Planner ({executor_gateway.descriptor}).')
else:
    exec_keys = [k.strip() for k in EXECUTOR_GEMINI_KEYS_INPUT.split(',') if k.strip()]
    if not exec_keys:
        sec_gem = _resolve_colab_secret('GEMINI_API_KEY') if '_resolve_colab_secret' in globals() else None
        if sec_gem:
            exec_keys.append(sec_gem)
    exec_hf = EXECUTOR_HF_TOKEN_INPUT.strip() or (_resolve_colab_secret('HF_TOKEN') if '_resolve_colab_secret' in globals() else None)
    executor_gateway = ModelGateway(
        provider=EXECUTOR_PROVIDER,
        model_name=EXECUTOR_GEMINI_MODEL,
        api_keys=exec_keys,
        local_model_path=EXECUTOR_LOCAL_MODEL_NAME,
        hf_token=exec_hf,
        quantization=EXECUTOR_QUANTIZATION
    )
    print(f'✅ Custom Executor Model configured: {executor_gateway.descriptor}')

pipeline.executor_gateway = executor_gateway
pipeline.gateway = executor_gateway
pipeline.model_manager = DualModelManager(pipeline.planner_gateway, pipeline.executor_gateway)
print(f'🧠 Planner: {pipeline.planner_gateway.descriptor}  |  ⚙️ Executor: {pipeline.executor_gateway.descriptor}')

## 5️⃣ Fail-Fast Code Synchronization & Diff Tracker
Clones/pulls source code and aborts immediately if clone fails or yields an empty folder (preventing hallucinated docs).

In [ ]:
#@title 5. Sync Code and Detect Changes { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 Current Git Commit: ({current_commit[:7] if current_commit else "Local"})')
if changed_files:
    print(f'🔍 Detected {len(changed_files)} changed files since last documented commit:')
    for f in changed_files[:15]:
        print('  •', f)
else:
    print('ℹ️ Full fresh repository inspection will run across all tracked source files.')

## 6️⃣ 100% Full-File Map-Reduce Generation, Quality Gate & RAG Indexing
Runs fresh full-repo extraction, Executor Map pass across 100% of files, Planner Master Blueprint, full-package authoring, Quality Gate (`L0`/`L1`), and persists `.rag_index/`.

In [ ]:
#@title 6. Execute 100% Full-Codebase Scan & Multi-Agent Generation { display-mode: "form" }
FORCE_FULL_GENERATION = True #@param {type:"boolean"}

def _show_map_progress(idx, total, module_name, files_desc):
    if idx == 1 or idx == total or idx % 5 == 0:
        print(f'  [{idx}/{total}] Inspecting `{module_name}`: {files_desc[:90]}')

target_changes = None if FORCE_FULL_GENERATION else changed_files
run_stats = pipeline.run_multi_agent_generation(
    changed_files=target_changes,
    force_full=FORCE_FULL_GENERATION,
    progress_callback=_show_map_progress
)

if current_commit:
    pipeline.save_state(current_commit)

print('='*70)
print('📊 Documentation Pipeline Telemetry:')
print(json.dumps(run_stats, indent=2, ensure_ascii=False))
print('='*70)

## 7️⃣ Documentation Portal & Cloudflare Tunnel
Build and launch interactive MkDocs Material site and push docs hub updates if configured.

In [ ]:
#@title 7. Launch Portal and Push to Remote Docs Repo { display-mode: "form" }
tunnel_url = pipeline.serve_and_tunnel(port=8000)

if tunnel_url:
    print('='*70)
    print('🎉 Interactive Documentation Portal is live at:')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ Server running locally, check tunnel logs.')

sync_docs_res = pipeline.sync_docs_repo()
print('📡 Remote Docs Sync:', sync_docs_res)

## 8️⃣ Hybrid BM25 Q&A Assistant (`.rag_index/`)
Query docs + codebase FileFactCards + DB schemas + API catalogs stored in `project_dir/.rag_index/`.

In [ ]:
#@title 8. Query Project Documentation & Verified Codebase Facts { display-mode: "form" }
QUESTION = "What is the system architecture, database schema, and core APIs?" #@param {type:"string"}
AUDIENCE = "Developer" #@param ["Developer", "Management", "End-User"]

chat_engine = pipeline.get_chat_engine()
chunks_loaded = chat_engine.load_documents(prefer_saved_index=True)
print(f'📚 Loaded {chunks_loaded} indexed knowledge chunks from `{chat_engine.index_dir}`')
print(f'🔍 Querying for: "{QUESTION}" (Audience: {AUDIENCE})...\n')

answer = chat_engine.ask(QUESTION, audience=AUDIENCE)
print('='*70)
print(answer)
print('='*70)